In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import io

# The parent page where the links live
base_page = "https://sec.gov.ng/for-operators/keep-track-of-capital-market-data/net-asset-value-data/monthly-net-asset-value-for-collective-investment-schemes/2026-monthly-nav-for-cis/"
# The root domain to fix the links
root_domain = "https://sec.gov.ng"

def get_latest_sec_data():
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(base_page, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all Excel links
        links = [a['href'] for a in soup.find_all('a', href=True) if a['href'].endswith('.xlsx')]
        
        if not links:
            return "No files found. Check if the SEC has updated the 2026 folder."
        
        # FIX: Join the relative path with the root domain
        latest_link = urljoin(root_domain, links[0])
        print(f"Success! Fetching from: {latest_link}")
        
        # Download the file content into memory
        file_response = requests.get(latest_link, headers=headers)
        
        # Read the Excel file from memory (io.BytesIO)
        # We skip 3 rows because SEC files usually have titles at the top
        df = pd.read_excel(io.BytesIO(file_response.content), skiprows=3)
        
        # Basic Cleaning
        df = df.dropna(how='all').dropna(axis=1, how='all')
        
        return df
    
    except Exception as e:
        return f"An error occurred: {e}"

# Execute
latest_funds_df = get_latest_sec_data()

# Show the first few rows to verify it worked
if isinstance(latest_funds_df, pd.DataFrame):
    print("Data Loaded Successfully!")
    display(latest_funds_df.head(10))
else:
    print(latest_funds_df)

Success! Fetching from: https://sec.gov.ng/documents/1453/Monthly_Update_on_Registered_Mutual_Funds_as_at_February_2026.xlsx
Data Loaded Successfully!


,EQUITY BASED FUNDS,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,1,Afrinvest Equity Fund,Afrinvest Asset Management Ltd.,8.666992e+09,1.982095e+07,2.721236e+09,11600781.25,2.729456e+09,6.118875e+09,0.060914,...,0.483133,0.001278,0.300763,773.270356,232.571147,773.2700,778.520,1695.0,9.259359e+06,1.173600e+07
1,2,Anchoria Equity Fund,Anchoria Asset Management Limited,1.672233e+09,2.805256e+06,0.000000e+00,1911774.35,8.934815e+05,1.389170e+09,0.013829,...,0.238794,0.001111,0.000519,57.314074,0.029757,493.7300,499.850,513.0,2.985107e+07,3.002572e+07
2,3,ARM Aggressive Growth Fund,ARM Investment Managers Limited,9.038494e+09,9.967616e+06,1.363800e+09,18828751.00,1.354939e+09,8.873618e+09,0.088337,...,0.292648,0.001641,0.118124,64.851723,7.660555,64.5274,66.473,9277.0,1.576392e+08,1.768722e+08
3,4,AXA Mansard Equity Income Fund,AXA Mansard Investments Limited,1.181413e+09,7.132127e+06,0.000000e+00,2264661.19,4.867465e+06,1.252759e+09,0.012471,...,0.457529,0.001240,0.002666,315.536049,0.841138,315.5360,315.536,2402.0,4.696752e+06,5.786763e+06
4,5,CardinalStone Equity Fund,CardinalStone Asset Mgt. Limited,4.408211e+09,1.055504e+08,5.765105e+08,5456196.09,6.766047e+08,3.503088e+09,0.034873,...,0.338492,0.001164,0.144301,2.325518,0.335574,2.3339,2.361,1511.0,1.756709e+09,2.016264e+09
5,6,Cowry Equity Fund,Cowry Treasurers Limited,4.638910e+08,1.711554e+06,0.000000e+00,1045084.27,6.664700e+05,3.337140e+08,0.003322,...,0.498842,0.002089,0.001332,277.172500,0.369318,277.1700,277.170,166.0,1.729652e+06,1.804597e+06
6,7,FBN Nigeria Smart Beta Equity Fund,FBN Capital Asset Mgt,4.851730e+09,1.162408e+07,6.996948e+08,4985574.46,7.063334e+08,3.293402e+09,0.032786,...,0.455160,0.001040,0.147385,568.325224,83.762792,564.3800,572.640,1964.0,6.857549e+06,8.432543e+06
7,8,Frontier Fund,SCM Capital Limited,6.264951e+08,2.711369e+06,1.101890e+08,1988865.32,1.109115e+08,5.294393e+08,0.005271,...,0.128726,0.003328,0.185597,298.484163,55.397866,298.4900,312.610,2475.0,2.002082e+06,2.002089e+06
8,9,Futureview Equity Fund,Futureview Asset Management Limited,1.134760e+08,5.957671e+05,1.767670e+07,2060737.79,1.621173e+07,9.972024e+07,0.000993,...,0.140729,0.018116,0.142516,410.398777,58.488363,402.8500,416.830,36.0,2.734085e+05,2.771787e+05
9,10,Guaranty Trust Equity Income Fund,Guaranty Trust Fund Managers Limited,1.289596e+10,2.652727e+07,1.470224e+09,14415009.40,1.482336e+09,5.856169e+09,0.058298,...,1.196436,0.001121,0.115243,5.180393,0.597004,5.1600,5.190,6698.0,1.364759e+09,2.482959e+09


In [5]:
import pandas as pd
import numpy as np

def clean_sec_data(df):
    # 1. Drop any completely empty rows/cols
    df = df.dropna(how='all').dropna(axis=1, how='all')
    
    # 2. SEC files usually have 15-20 columns. Let's map the core ones we need.
    # We identify columns by index because names like 'Unnamed: 1' are unreliable.
    df_clean = df.copy()
    
    # Standard column mapping for the February 2026 file
    # Index 0: S/N, Index 1: Fund Name, Index 2: Manager, Index 8: NAV, Index 12: Yield
    cols_to_keep = {
        df.columns[1]: 'Fund_Name',
        df.columns[2]: 'Fund_Manager',
        df.columns[8]: 'NAV_Per_Unit',
        df.columns[12]: 'Yield_YTD'
    }
    
    df_clean = df_clean.rename(columns=cols_to_keep)
    
    # 3. Fill the Manager names (they are merged in Excel)
    df_clean['Fund_Manager'] = df_clean['Fund_Manager'].ffill()
    
    # 4. Clean numeric values (remove strings/garbage)
    df_clean['Yield_YTD'] = pd.to_numeric(df_clean['Yield_YTD'], errors='coerce')
    df_clean['NAV_Per_Unit'] = pd.to_numeric(df_clean['NAV_Per_Unit'], errors='coerce')
    
    # 5. Remove header rows (rows where Fund_Name is actually a category title like 'EQUITY BASED')
    # Valid funds usually have a numeric Yield or a proper NAV
    df_final = df_clean[df_clean['Yield_YTD'].notna()].copy()
    
    return df_final

# Apply the cleaning
try:
    final_investment_df = clean_sec_data(latest_funds_df)
    print(f"Success! Processed {len(final_investment_df)} active funds.")
    
    # Show top 10 for your 5k-50k project
    display(final_investment_df.sort_values('Yield_YTD', ascending=False).head(10))
except Exception as e:
    print(f"Cleaning failed: {e}")

Success! Processed 209 active funds.


,EQUITY BASED FUNDS,Fund_Name,Fund_Manager,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,NAV_Per_Unit,Unnamed: 9,...,Yield_YTD,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
18,19,Zedcrest Equity Fund,Zedcrest Investment Managers Limited,1.546842e+09,3.248503e+07,0.000000e+00,1479487.50,3.100554e+07,4.055462e+08,0.004037,...,3.012408,0.000909,0.019054,1.468781,0.027987,1.4688,1.4688,700.0,1.107869e+09,1.107869e+09
230,2,Clean Energy Fund,Fundco Capital Managers Limited,3.549478e+09,1.843710e+08,0.000000e+00,13439564.84,1.709315e+08,4.192943e+09,0.205151,...,2.708101,0.000864,0.010994,108.803153,1.196171,123.3000,123.3000,12.0,3.318520e+07,1.428989e+08
170,153,Coronation Balanced Fund,Coronation Asset Management Limited,4.220958e+09,3.055409e+07,7.274177e+08,7766541.87,7.502052e+08,1.490793e+09,0.015771,...,1.860880,0.001821,0.175899,2.599260,0.457207,2.5696,2.6183,4264.0,6.603115e+08,1.640844e+09
171,154,Cowry Balanced Fund,Cowry Treasurers Limited,5.992585e+08,2.236349e+06,0.000000e+00,1395869.99,8.404791e+05,2.371533e+08,0.002509,...,1.849295,0.002066,0.001244,632.200969,0.786349,248.5600,248.5600,187.0,9.639142e+05,1.068837e+06
94,89,Norrenberger Turbo Fixed Income Fund,Norrenberger Investment & Capital Mgt. Ltd.,4.732220e+09,3.329178e+07,3.561252e+07,6041631.48,6.286267e+07,1.825846e+09,0.007584,...,1.760339,0.001199,0.012473,104.972668,1.309310,104.9700,104.9700,466.0,1.774680e+07,4.801206e+07
202,178,Stanbic IBTC Imaan Fund,Stanbic IBTC Asset Mgt. Limited,1.400374e+10,3.005321e+07,2.081492e+09,47253344.72,2.064292e+09,7.281688e+09,0.085172,...,1.383201,0.002723,0.118954,1212.402013,144.219875,1204.7300,1217.6600,4569.0,7.173167e+06,1.431351e+07
9,10,Guaranty Trust Equity Income Fund,Guaranty Trust Fund Managers Limited,1.289596e+10,2.652727e+07,1.470224e+09,14415009.40,1.482336e+09,5.856169e+09,0.058298,...,1.196436,0.001121,0.115243,5.180393,0.597004,5.1600,5.1900,6698.0,1.364759e+09,2.482959e+09
15,16,Stanbic IBTC Aggressive Fund (Sub Fund),Stanbic IBTC Asset Mgt. Limited,2.626320e+09,7.004996e+06,3.824317e+08,7337956.79,3.820987e+08,1.970050e+09,0.019612,...,0.677993,0.002220,0.115587,15782.929756,1824.297872,15710.3300,15877.5800,53.0,1.462731e+05,2.094497e+05
76,71,CFG AM Fixed Income Naira Fund,CFG Assset Management Limited,4.058700e+08,3.590594e+07,0.000000e+00,4540219.32,3.136572e+07,1.257614e+09,0.005223,...,0.577865,0.002288,0.015807,1161.062767,18.352443,1161.0600,1161.0600,295.0,1.097805e+06,1.709076e+06
222,194,ARM Halal Balanced Fund,ARM Investment Managers Limited,5.966661e+09,3.429282e+07,1.066566e+09,63490214.00,1.037369e+09,6.325124e+09,0.073983,...,0.502094,0.006683,0.109186,129.815263,14.174012,129.1662,133.0607,3950.0,5.658400e+07,7.318810e+07


In [12]:
import pandas as pd
import numpy as np

def final_clean_and_simulate(df):
    # 1. Create a fresh copy
    df_fix = df.copy()
    
    # 2. Map by Index (SEC Standard Layout)
    # 0: S/N, 1: Fund Name, 2: Manager, 5: Net Asset Value, 6: Units Outstanding, 12: Yield
    # We use .iloc to ensure we get the right data regardless of the header name
    data = {
        'Fund_Name': df_fix.iloc[:, 1],
        'Manager': df_fix.iloc[:, 2],
        'Total_NAV': pd.to_numeric(df_fix.iloc[:, 5], errors='coerce'),
        'Units_Out': pd.to_numeric(df_fix.iloc[:, 6], errors='coerce'),
        'Yield_YTD': pd.to_numeric(df_fix.iloc[:, 12], errors='coerce')
    }
    
    new_df = pd.DataFrame(data)
    
    # 3. Calculate the Actual Unit Price
    # (Total NAV / Units Outstanding)
    new_df['Unit_Price'] = new_df['Total_NAV'] / new_df['Units_Out']
    
    # 4. Filter for valid funds (Price > 0 and Yield exists)
    # We exclude the institutional 'giant' prices to find retail funds
    clean_df = new_df[(new_df['Unit_Price'] > 0.1) & (new_df['Unit_Price'] < 10000)].copy()
    
    # 5. Simulation for your 5k and 50k budget
    clean_df['Units_for_N5,000'] = 5000 / clean_df['Unit_Price']
    clean_df['Units_for_N50,000'] = 50000 / clean_df['Unit_Price']
    
    # 6. Final Polish: Display as percentage
    clean_df['Yield_Percent'] = clean_df['Yield_YTD'] * 100
    
    return clean_df

# Execute
pd.options.display.float_format = '{:,.2f}'.format
try:
    my_portfolio_data = final_clean_and_simulate(latest_funds_df)
    print(f"Success! Found {len(my_portfolio_data)} retail funds.")
    display(my_portfolio_data.sort_values('Yield_Percent', ascending=False).head(15))
except Exception as e:
    print(f"Error: {e}. Check if your dataframe has at least 13 columns.")

Success! Found 68 retail funds.


,Fund_Name,Manager,Total_NAV,Units_Out,Yield_YTD,Unit_Price,"Units_for_N5,000","Units_for_N50,000",Yield_Percent
170,Coronation Balanced Fund,Coronation Asset Management Limited,"727,417,682.25","7,766,541.87",1.86,93.66,53.38,533.84,186.09
94,Norrenberger Turbo Fixed Income Fund,Norrenberger Investment & Capital Mgt. Ltd.,"35,612,519.58","6,041,631.48",1.76,5.89,848.25,"8,482.45",176.03
202,Stanbic IBTC Imaan Fund,Stanbic IBTC Asset Mgt. Limited,"2,081,492,335.94","47,253,344.72",1.38,44.05,113.51,"1,135.08",138.32
9,Guaranty Trust Equity Income Fund,Guaranty Trust Fund Managers Limited,"1,470,223,844.78","14,415,009.40",1.20,101.99,49.02,490.23,119.64
15,Stanbic IBTC Aggressive Fund (Sub Fund),Stanbic IBTC Asset Mgt. Limited,"382,431,658.05","7,337,956.79",0.68,52.12,95.94,959.38,67.80
222,ARM Halal Balanced Fund,ARM Investment Managers Limited,"1,066,566,347.00","63,490,214.00",0.50,16.80,297.64,"2,976.38",50.21
0,Afrinvest Equity Fund,Afrinvest Asset Management Ltd.,"2,721,235,609.08","11,600,781.25",0.48,234.57,21.32,213.15,48.31
16,Stanbic IBTC Nigerian Equity Fund,Stanbic IBTC Asset Mgt. Limited,"6,597,398,359.63","134,773,110.95",0.48,48.95,102.14,"1,021.41",47.93
14,Paramount Equity Fund,Chapel Hill Denham Mgt. Limited,"2,468,433,671.53","18,866,143.57",0.46,130.84,38.21,382.15,45.77
6,FBN Nigeria Smart Beta Equity Fund,FBN Capital Asset Mgt,"699,694,846.80","4,985,574.46",0.46,140.34,35.63,356.27,45.52


In [14]:
%%writefile invest_app.py
import streamlit as st
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import io

st.set_page_config(page_title="Nigeria Mutual Fund Tracker", layout="wide")

st.title("🇳🇬 Real-Time Mutual Fund ROI Tracker")
st.markdown("Automated Investment Analysis by Bamidele Adedeji")

# --- DATA SCRAPING & CLEANING ---
@st.cache_data(ttl=86400)
def get_and_clean_data():
    base_url = "https://sec.gov.ng/for-operators/keep-track-of-capital-market-data/net-asset-value-data/monthly-net-asset-value-for-collective-investment-schemes/2026-monthly-nav-for-cis/"
    root = "https://sec.gov.ng"
    response = requests.get(base_url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(response.text, 'html.parser')
    links = [urljoin(root, a['href']) for a in soup.find_all('a', href=True) if a['href'].endswith('.xlsx')]
    
    file_res = requests.get(links[0])
    df = pd.read_excel(io.BytesIO(file_res.content), skiprows=3)
    
    # Using our perfected index-based cleaning
    data = {
        'Fund_Name': df.iloc[:, 1],
        'Manager': df.iloc[:, 2],
        'Total_NAV': pd.to_numeric(df.iloc[:, 5], errors='coerce'),
        'Units_Out': pd.to_numeric(df.iloc[:, 6], errors='coerce'),
        'Yield_YTD': pd.to_numeric(df.iloc[:, 12], errors='coerce')
    }
    new_df = pd.DataFrame(data)
    new_df['Unit_Price'] = new_df['Total_NAV'] / new_df['Units_Out']
    
    # Filter for Retail
    clean_df = new_df[(new_df['Unit_Price'] > 0.1) & (new_df['Unit_Price'] < 10000)].copy()
    clean_df['Yield_Percent'] = clean_df['Yield_YTD'] * 100
    return clean_df

# --- APP INTERFACE ---
try:
    df = get_and_clean_data()
    
    st.sidebar.header("Settings")
    budget = st.sidebar.slider("Monthly Investment (₦)", 5000, 50000, 10000, 5000)
    
    df['Units_Purchased'] = budget / df['Unit_Price']
    
    st.metric("Funds Found", len(df))
    st.dataframe(df.sort_values('Yield_Percent', ascending=False), use_container_width=True)
    
    st.subheader("Performance Chart")
    st.bar_chart(df.sort_values('Yield_Percent', ascending=False).head(10), x="Fund_Name", y="Yield_Percent")
    
except Exception as e:
    st.error(f"Error loading data: {e}")

Overwriting invest_app.py


In [15]:
%%writefile requirements.txt
streamlit
pandas
requests
beautifulsoup4
openpyxl

Overwriting requirements.txt


In [16]:
import os
print(os.getcwd())

C:\Users\user
